<a href="https://colab.research.google.com/github/Samujjalborah/python-learning-journey/blob/main/Day_12_Used_Car_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 12 - Used Car Data Preprocessing

## Objective

This notebook demonstrates a complete data preprocessing workflow using the Used Car Resale dataset. The preprocessing steps include data inspection, missing value checking, train-test splitting, outlier handling, categorical encoding, feature scaling, and exporting the processed datasets for machine learning.

**Developed by:** Samujjal Borah

In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
df = pd.read_csv("car_resale_data.csv")
df.head()


,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


In [22]:
print("Dataset Shape:", df.shape)


Dataset Shape: (320, 15)


Dataset Information

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car_ID              320 non-null    object 
 1   Brand               320 non-null    object 
 2   Year                320 non-null    int64  
 3   Mileage_Km          320 non-null    int64  
 4   Engine_CC           320 non-null    int64  
 5   Power_BHP           320 non-null    float64
 6   Fuel_Type           320 non-null    object 
 7   Transmission        320 non-null    object 
 8   City                320 non-null    object 
 9   Seller_Type         320 non-null    object 
 10  Condition           320 non-null    object 
 11  Previous_Owners     320 non-null    int64  
 12  Accidents_Reported  320 non-null    int64  
 13  Service_Score       320 non-null    int64  
 14  Resale_Price_Lakh   320 non-null    float64
dtypes: float64(2), int64(6), object(7)
memory usage: 37.6+ KB

Summary Statistics

In [24]:
df.describe()

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
count,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000
mean,2019.537500,74110.203125,1346.703125,150.489688,1.668750,0.243750,76.203125,4.963031
std,3.341367,38885.260771,543.408160,36.665353,0.865369,0.528164,12.745864,3.359259
min,2014.000000,700.000000,600.000000,51.400000,1.000000,0.000000,55.000000,1.200000
25%,2017.000000,46323.250000,1004.750000,128.450000,1.000000,0.000000,64.750000,2.277500
50%,2020.000000,72718.500000,1303.000000,150.750000,1.000000,0.000000,77.000000,4.610000
75%,2022.000000,97951.500000,1635.250000,171.475000,2.000000,0.000000,87.000000,6.835000
max,2025.000000,320000.000000,5000.000000,390.000000,4.000000,2.000000,98.000000,28.500000


Missing Values

In [ ]:
df.isnull().sum()

,0
Car_ID,0
Brand,0
Year,0
Mileage_Km,0
Engine_CC,0
Power_BHP,0
Fuel_Type,0
Transmission,0
City,0
Seller_Type,0


Separate Features and Target

In [25]:
X = df.drop(columns=["Resale_Price_Lakh"])
y = df["Resale_Price_Lakh"]

Train-Test Split

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create copies to avoid SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()

Check the Split

In [27]:
print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Training Features: (256, 14)
Testing Features: (64, 14)
Training Target: (256,)
Testing Target: (64,)


Define Numerical Columns

In [28]:
numerical_columns = [
    "Year",
    "Mileage_Km",
    "Engine_CC",
    "Power_BHP",
    "Previous_Owners",
    "Accidents_Reported",
    "Service_Score"
]

Handle Outliers (IQR Method)

In [29]:
for col in numerical_columns:

    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - (1.5 * IQR)
    upper = Q3 + (1.5 * IQR)

    mask = X_train[col].between(lower, upper)

    X_train = X_train.loc[mask]
    y_train = y_train.loc[mask]

Check the New Training Size

In [30]:
print("Training Features after outlier removal:", X_train.shape)
print("Training Target after outlier removal:", y_train.shape)

Training Features after outlier removal: (198, 14)
Training Target after outlier removal: (198,)


Define Categorical Columns

In [31]:
categorical_columns = [
    "Brand",
    "Fuel_Type",
    "Transmission",
    "City",
    "Seller_Type",
    "Condition"
]

One-Hot Encoding

In [32]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded_train = encoder.fit_transform(
    X_train[categorical_columns]
)

encoded_test = encoder.transform(
    X_test[categorical_columns]
)

Define Numerical Columns for Scaling

In [33]:
numerical_columns = [
    "Year",
    "Mileage_Km",
    "Engine_CC",
    "Power_BHP",
    "Previous_Owners",
    "Accidents_Reported",
    "Service_Score"
]

In [35]:
# Convert numerical columns to float before scaling
X_train[numerical_columns] = X_train[numerical_columns].astype(float)
X_test[numerical_columns] = X_test[numerical_columns].astype(float)

Min-Max Scaling

In [36]:
scaler = MinMaxScaler()

X_train.loc[:, numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)

X_test.loc[:, numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)

Verification

In [ ]:
print(X_train.head())

print(X_train.shape)

      Car_ID      Year  Mileage_Km  Engine_CC  Power_BHP  Previous_Owners  \
247  CAR0248  0.272727    0.732086   0.726101   0.686486              0.0   
302  CAR0303  0.727273    0.486381   0.293770   0.220270              0.0   
259  CAR0260  0.818182    0.177540   0.630505   0.670946              0.0   
129  CAR0130  0.545455    0.571514   0.244898   0.133108              0.5   
305  CAR0306  0.000000    0.630618   0.528464   0.114865              0.5   

     Accidents_Reported  Service_Score  Brand_Honda  Brand_Hyundai  ...  \
247                 0.0       1.000000          0.0            0.0  ...   
302                 0.0       0.883721          0.0            0.0  ...   
259                 0.0       0.395349          0.0            0.0  ...   
129                 0.0       0.720930          1.0            0.0  ...   
305                 0.0       0.883721          1.0            0.0  ...   

     City_Mumbai  City_Pune  Seller_Type_Certified Dealer  Seller_Type_Dealer  \
247  

Process the Test Dataset

In [37]:
encoded_train_df = pd.DataFrame(
    encoded_train,
    columns=encoder.get_feature_names_out(categorical_columns),
    index=X_train.index
)

encoded_test_df = pd.DataFrame(
    encoded_test,
    columns=encoder.get_feature_names_out(categorical_columns),
    index=X_test.index
)

X_train_final = pd.concat(
    [X_train.drop(columns=categorical_columns), encoded_train_df],
    axis=1
)

X_test_final = pd.concat(
    [X_test.drop(columns=categorical_columns), encoded_test_df],
    axis=1
)

Verification

In [38]:
print("Training Dataset:", X_train_final.shape)
print("Testing Dataset:", X_test_final.shape)

X_train_final.head()

Training Dataset: (198, 42)
Testing Dataset: (64, 42)


,Car_ID,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Brand_Honda,Brand_Hyundai,...,City_Mumbai,City_Pune,Seller_Type_Certified Dealer,Seller_Type_Dealer,Seller_Type_Individual,Condition_Excellent,Condition_Fair,Condition_Good,Condition_Poor,Condition_Very Good
132,CAR0133,0.363636,0.381017,0.315789,0.563370,0.0,0.0,0.372093,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
234,CAR0235,0.090909,0.661739,0.364662,0.589136,0.0,0.0,0.348837,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
312,CAR0313,0.818182,0.387463,0.509667,0.587744,0.5,0.0,0.558140,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
232,CAR0233,0.181818,0.576138,0.583781,0.399025,0.5,0.0,0.534884,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
124,CAR0125,0.909091,0.307055,0.351235,0.228412,0.5,0.0,0.953488,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Final Export

In [40]:
train_processed = X_train_final.copy()
train_processed["Resale_Price_Lakh"] = y_train

test_processed = X_test_final.copy()
test_processed["Resale_Price_Lakh"] = y_test

train_processed.to_csv("Processed_Train_Dataset.csv", index=False)
test_processed.to_csv("Processed_Test_Dataset.csv", index=False)

print("✅ Processed datasets exported successfully!")

✅ Processed datasets exported successfully!


In [41]:
import os
os.listdir()

['.config',
 'Processed_Test_Dataset.csv',
 'Processed_Train_Dataset.csv',
 'car_resale_data.csv',
 'sample_data']

# Observations

- The original dataset contains 320 records and 15 columns.
- No missing values were found in the dataset.
- The dataset was divided into training and testing sets before preprocessing to avoid data leakage.
- Outliers in the training data were handled using the IQR method.
- Categorical variables were converted into numerical features using One-Hot Encoding.
- Numerical features were scaled using Min-Max Scaling.
- The encoder and scaler were fitted only on the training data and then applied to the test data.
- The processed training and testing datasets were exported successfully as CSV files.



# Conclusion

The used car resale dataset was successfully preprocessed for machine learning. The workflow included data inspection, train-test splitting, outlier handling, categorical encoding, numerical scaling, and exporting the processed datasets. The final datasets are clean, transformed, and ready for model training and evaluation.

One extra verification

In [42]:
print("Training Dataset Shape:", train_processed.shape)
print("Testing Dataset Shape:", test_processed.shape)

train_processed.head()

Training Dataset Shape: (198, 43)
Testing Dataset Shape: (64, 43)


,Car_ID,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Brand_Honda,Brand_Hyundai,...,City_Pune,Seller_Type_Certified Dealer,Seller_Type_Dealer,Seller_Type_Individual,Condition_Excellent,Condition_Fair,Condition_Good,Condition_Poor,Condition_Very Good,Resale_Price_Lakh
132,CAR0133,0.363636,0.381017,0.315789,0.563370,0.0,0.0,0.372093,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.26
234,CAR0235,0.090909,0.661739,0.364662,0.589136,0.0,0.0,0.348837,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.23
312,CAR0313,0.818182,0.387463,0.509667,0.587744,0.5,0.0,0.558140,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,7.09
232,CAR0233,0.181818,0.576138,0.583781,0.399025,0.5,0.0,0.534884,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,2.69
124,CAR0125,0.909091,0.307055,0.351235,0.228412,0.5,0.0,0.953488,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,5.71


In [43]:
print(train_processed.columns.tolist())

['Car_ID', 'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Previous_Owners', 'Accidents_Reported', 'Service_Score', 'Brand_Honda', 'Brand_Hyundai', 'Brand_Kia', 'Brand_Mahindra', 'Brand_Maruti', 'Brand_Renault', 'Brand_Skoda', 'Brand_Tata', 'Brand_Toyota', 'Brand_Volkswagen', 'Fuel_Type_CNG', 'Fuel_Type_Diesel', 'Fuel_Type_Electric', 'Fuel_Type_Petrol', 'Transmission_Automatic', 'Transmission_Manual', 'City_Ahmedabad', 'City_Bengaluru', 'City_Chandigarh', 'City_Delhi', 'City_Hyderabad', 'City_Jaipur', 'City_Kochi', 'City_Lucknow', 'City_Mumbai', 'City_Pune', 'Seller_Type_Certified Dealer', 'Seller_Type_Dealer', 'Seller_Type_Individual', 'Condition_Excellent', 'Condition_Fair', 'Condition_Good', 'Condition_Poor', 'Condition_Very Good', 'Resale_Price_Lakh']
